## Defining architechure and train function


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

transform_train = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transform_test = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.ImageFolder('content/fer2013plus/fer2013plus/fer2013/train/', transform=transform_train)
test_dataset  = datasets.ImageFolder('content/fer2013plus/fer2013plus/fer2013/test/', transform=transform_test)

train_loader = DataLoader(
    train_dataset, 
    batch_size=512, 
    shuffle=True, 
    num_workers=8, 
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=512, 
    shuffle=False,
    num_workers=8, 
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)

class FER_DCNN(nn.Module):
    def __init__(self):
        super(FER_DCNN, self).__init__()

        # Helper to create a block with 2 convs per block
        def conv_block(in_f, out_f, dropout_rate):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ELU(),
                nn.Conv2d(out_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ELU(),
                nn.MaxPool2d(2),
                nn.Dropout(dropout_rate)
            )

        # 1. Conv block count: 3 sets
        # 2. Filter counts: 64, 128, 256
        # 3. Dropout (conv): Rate 0.3 for all blocks
        self.features = nn.Sequential(
            conv_block(1, 64, 0.3),   # Block 1
            conv_block(64, 128, 0.3),  # Block 2
            conv_block(128, 256, 0.3)  # Block 3
        )

        # 4. Dense layers: 1 dense, 128 neurons
        # Spatial size logic: 48 -> 24 -> 12 -> 6 (after 3 MaxPools)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Dropout(0.4),          # 5. Dropout (dense): Rate 0.4
            nn.Linear(128, 8)         # Keeping 8 output neurons as per your code
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FER_DCNN().to(device)


class_counts = np.array([len([x for x in train_dataset.targets if x == i]) for i in range(8)])
class_weights = 1.0 / class_counts
class_weights = torch.tensor(class_weights / class_weights.sum(), dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        
        if torch.cuda.is_available():
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0  # Initialize total_loss

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            if torch.cuda.is_available():
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    outputs = model(images)
                    loss = criterion(outputs, labels) # Calculate loss
            else:
                outputs = model(images)
                loss = criterion(outputs, labels) # Calculate loss
                
            _, preds = torch.max(outputs, 1)

            total_loss += loss.item() # Accumulate loss
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total, total_loss / len(loader)

## Training Model

In [2]:
# Training loop
best_acc, best_val_loss = 0, float('inf')
patience, no_improve_count, epoch = 10, 0, 0

while True:
    train_loss = train_epoch(model, train_loader)
    val_acc, val_loss = evaluate(model, test_loader)

    scheduler.step(val_loss)   # Fix: step on val_loss, not train_loss

    if val_acc > best_acc + 1e-4:
        best_acc = val_acc
        no_improve_count = 0
        torch.save(model.state_dict(), "best_fer_model.pth")
    else:
        no_improve_count += 1

    print(f"Epoch {epoch+1}: TrainLoss={train_loss:.4f}  ValLoss={val_loss:.4f}  "
          f"ValAcc={val_acc:.4f}  NoImprove={no_improve_count}/{patience}")

    if no_improve_count >= patience:
        print(f"Converged at epoch {epoch+1}. Best Val Acc: {best_acc:.4f}")
        break

    epoch += 1

Epoch 1: TrainLoss=1.9685  ValLoss=1.9038  ValAcc=0.2692  NoImprove=0/10
Epoch 2: TrainLoss=1.7261  ValLoss=1.7438  ValAcc=0.3282  NoImprove=0/10
Epoch 3: TrainLoss=1.5402  ValLoss=1.5143  ValAcc=0.4591  NoImprove=0/10
Epoch 4: TrainLoss=1.4089  ValLoss=1.4018  ValAcc=0.4746  NoImprove=0/10
Epoch 5: TrainLoss=1.3226  ValLoss=1.5094  ValAcc=0.4282  NoImprove=1/10
Epoch 6: TrainLoss=1.2732  ValLoss=1.3813  ValAcc=0.4746  NoImprove=2/10
Epoch 7: TrainLoss=1.1822  ValLoss=1.2026  ValAcc=0.5490  NoImprove=0/10
Epoch 8: TrainLoss=1.1621  ValLoss=1.2829  ValAcc=0.4884  NoImprove=1/10
Epoch 9: TrainLoss=1.1199  ValLoss=1.1580  ValAcc=0.5568  NoImprove=0/10
Epoch 10: TrainLoss=1.0580  ValLoss=1.1335  ValAcc=0.5914  NoImprove=0/10
Epoch 11: TrainLoss=1.0281  ValLoss=1.1777  ValAcc=0.5485  NoImprove=1/10
Epoch 12: TrainLoss=0.9894  ValLoss=1.2698  ValAcc=0.5332  NoImprove=2/10
Epoch 13: TrainLoss=0.9553  ValLoss=1.1109  ValAcc=0.5754  NoImprove=3/10
Epoch 14: TrainLoss=0.9237  ValLoss=1.0550  Val